# Feature Engineering & Preprocessing Pipeline
## Leak-free Scikit-Learn workflow for a mixed tabular dataset

**Goal:** build a reusable preprocessing and modeling pipeline that handles missing values, numerical scaling, categorical encoding, correlation analysis, and feature-importance analysis without leaking information from the test set.

### Dataset
This notebook uses the **Adult Census Income** dataset from OpenML. It contains numerical and categorical columns and predicts whether annual income exceeds `$50K`.

> The first data-loading run requires an internet connection because `fetch_openml` downloads the dataset. The notebook uses a fixed random seed for reproducibility.

### Leakage-prevention rules
1. Split raw features into training and test sets **before fitting any preprocessing step**.
2. Fit imputers, scalers, encoders, and the model only through the training data.
3. Keep preprocessing inside a Scikit-Learn `Pipeline` so test data receives only the transformations learned from training data.
4. Use training data only for correlation and feature-importance analysis.

## 1. Imports and reproducibility

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.feature_selection import mutual_info_classif

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## 2. Load the mixed-type dataset

The OpenML Adult dataset includes both numeric and categorical predictors. Missing values may be represented as `?`; convert those markers to `NaN` so the imputer can handle them.

In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)

df = adult.frame.copy()
target_column = "class"

# Convert OpenML missing-value markers to proper missing values.
df = df.replace("?", np.nan)

X = df.drop(columns=[target_column])
y = df[target_column].astype(str).str.strip()

print("Dataset shape:", df.shape)
print("\\nTarget distribution:")
display(y.value_counts(dropna=False).to_frame("count"))
display(df.head())

## 3. Inspect columns and split before preprocessing

Column types are identified from the raw feature table. The train/test split happens before any imputer, scaler, or encoder is fitted.

In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\\nTraining rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 4. Build reusable preprocessing pipelines

- **Numerical pipeline:** median imputation, then standard scaling.
- **Categorical pipeline:** most-frequent imputation, then one-hot encoding.
- `handle_unknown="ignore"` allows the test set to contain a category not seen during training.
- `ColumnTransformer` applies each pipeline to the correct columns.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

preprocessor

## 5. Create the complete model pipeline and train it

The preprocessing object is part of the model pipeline. Calling `.fit()` on `X_train` learns imputation values, scaling parameters, category levels, and model parameters from training data only.

In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred))

## 6. Evaluate the model

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=45,
    cmap="Blues"
)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

## 7. Correlation analysis (training data only)

Pearson correlation is calculated only from the **training split's numeric columns**. This is an exploratory analysis, not a transformation used to fit the model. Correlation indicates linear association; it does not prove causation.

The heatmap is kept to numeric columns because one-hot encoded categories can create a very wide matrix.

In [ ]:
train_numeric_for_corr = X_train[numeric_features].copy()
correlation_matrix = train_numeric_for_corr.corr(method="pearson")

plt.figure(figsize=(10, 7))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Numeric Feature Correlation — Training Set Only")
plt.tight_layout()
plt.show()

# Show highly correlated numeric feature pairs (absolute correlation >= 0.70).
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
high_corr_pairs = (
    upper_triangle.stack()
    .loc[lambda values: values.abs() >= 0.70]
    .sort_values(key=np.abs, ascending=False)
)
display(high_corr_pairs.to_frame("pearson_correlation"))

## 8. Feature importance — Random Forest

The fitted model's feature importances correspond to the columns produced by the fitted `ColumnTransformer`. We retrieve the names from the already-fitted encoder, so the names align with the transformed matrix.

In [ ]:
fitted_preprocessor = model_pipeline.named_steps["preprocessor"]
fitted_classifier = model_pipeline.named_steps["classifier"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_classifier.feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(20))

top_n = 20
plot_data = importance_df.head(top_n).sort_values("importance")
plt.figure(figsize=(10, 8))
plt.barh(plot_data["feature"], plot_data["importance"])
plt.xlabel("Random Forest feature importance")
plt.title(f"Top {top_n} Features")
plt.tight_layout()
plt.show()

## 9. Alternative: mutual information ranking

Mutual information can identify non-linear statistical dependence between each transformed feature and the target. It is calculated on the **transformed training set only**. This is an analysis step; it does not change the fitted model.

The transformed data is sparse because of one-hot encoding, so it is converted to a dense array for this moderate-sized teaching example. For very large datasets, consider sparse-compatible alternatives or a smaller feature sample.

In [ ]:
X_train_transformed = fitted_preprocessor.transform(X_train)

# Convert sparse output to dense for mutual_info_classif in this example.
if hasattr(X_train_transformed, "toarray"):
    X_train_for_mi = X_train_transformed.toarray()
else:
    X_train_for_mi = np.asarray(X_train_transformed)

mi_scores = mutual_info_classif(
    X_train_for_mi,
    y_train,
    random_state=RANDOM_STATE
)

mi_df = (
    pd.DataFrame({
        "feature": feature_names,
        "mutual_information": mi_scores
    })
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(mi_df.head(20))

## 10. Reuse the pipeline on new data

For future data, keep the same raw column names and pass the unprocessed DataFrame to `predict` or `predict_proba`. Do **not** manually impute, scale, or encode it: the fitted pipeline applies the training-learned transformations automatically.

In [ ]:
# Example: predict a few held-out rows using the complete pipeline.
sample_predictions = model_pipeline.predict(X_test.head(5))

display(pd.DataFrame({
    "actual": y_test.head(5).values,
    "predicted": sample_predictions
}, index=X_test.head(5).index))

## 11. Leakage checklist and conclusion

- [x] Raw data was split into training and test sets before fitting transformations.
- [x] Missing values are handled inside the pipelines.
- [x] Numeric scaling is fitted only on training data.
- [x] One-hot categories are learned only from training data.
- [x] Unknown test-time categories are safely ignored.
- [x] Model fitting and preprocessing are combined in one reusable `Pipeline`.
- [x] Correlation and feature-importance analysis use training data only.
- [x] Final performance is reported on the held-out test set.

**Conclusion:** This notebook demonstrates a reusable, leak-free Scikit-Learn workflow for mixed numerical and categorical tabular data. The same pattern can be adapted to credit-risk, housing, or medical datasets by changing the data-loading section, target column, and task-specific estimator.